# --- PHẦN 1: KHÁM PHÁ VÀ XÂY DỰNG PIPELINE TRÊN 1 NGƯỜI (SUBJECT 2) ---

In [3]:
import os
import pickle
import numpy as np
import pandas as pd

# 1. ĐƯỜNG DẪN ĐẾN FILE DỮ LIỆU
# Giả sử bạn đã giải nén thư mục WESAD gốc và để S2 ở đây
file_path = '../data/raw/WESAD/S2/S2.pkl'

print("Đang đọc file dữ liệu, vui lòng chờ vài giây...")

# 2. ĐỌC FILE PICKLE (.pkl)
try:
    with open(file_path, 'rb') as file:
        # Lưu ý: WESAD được nén bằng Python 2, bắt buộc phải dùng encoding='latin1'
        data = pickle.load(file, encoding='latin1')
    
    print(f"✅ Đã đọc thành công dữ liệu của Subject: {data['subject']}")
    
    # 3. TRÍCH XUẤT TÍN HIỆU CỔ TAY (WRIST - ĐỒNG HỒ EMPATICA E4)
    # Vì đề tài của bạn hướng tới thiết bị đeo tay (wearable), ta sẽ dùng data từ cổ tay
    wrist_data = data['signal']['wrist']
    
    acc = wrist_data['ACC']    # Gia tốc kế (Tần số lấy mẫu: 32 Hz)
    bvp = wrist_data['BVP']    # Thể tích máu/Nhịp tim (Tần số lấy mẫu: 64 Hz)
    eda = wrist_data['EDA']    # Hoạt động điện da/Mồ hôi (Tần số lấy mẫu: 4 Hz)
    temp = wrist_data['TEMP']  # Nhiệt độ da (Tần số lấy mẫu: 4 Hz)
    
    # 4. TRÍCH XUẤT NHÃN (LABELS)
    # Nhãn được hệ thống đo ở tần số cực cao của thiết bị ngực (700 Hz)
    labels = data['label']
    
    # 5. IN RA KÍCH THƯỚC (SHAPE) ĐỂ KHÁM PHÁ DỮ LIỆU
    print("-" * 50)
    print("📊 THỐNG KÊ KÍCH THƯỚC CÁC MẢNG DỮ LIỆU THÔ:")
    print(f" - Gia tốc (ACC) : {acc.shape} (Gồm 3 trục X, Y, Z)")
    print(f" - Nhịp tim (BVP): {bvp.shape}")
    print(f" - Điện da (EDA) : {eda.shape}")
    print(f" - Nhiệt độ (TEMP): {temp.shape}")
    print(f" - Mảng Nhãn (Label): {labels.shape}")
    print("-" * 50)
    print("🎯 Hoàn thành Cell 1! Dữ liệu đã sẵn sàng trên RAM.")

except FileNotFoundError:
    print(f"❌ LỖI: Không tìm thấy file tại '{file_path}'. Bạn hãy kiểm tra lại xem đã thả đúng file S2.pkl vào thư mục chưa nhé!")

Đang đọc file dữ liệu, vui lòng chờ vài giây...
✅ Đã đọc thành công dữ liệu của Subject: S2
--------------------------------------------------
📊 THỐNG KÊ KÍCH THƯỚC CÁC MẢNG DỮ LIỆU THÔ:
 - Gia tốc (ACC) : (194528, 3) (Gồm 3 trục X, Y, Z)
 - Nhịp tim (BVP): (389056, 1)
 - Điện da (EDA) : (24316, 1)
 - Nhiệt độ (TEMP): (24316, 1)
 - Mảng Nhãn (Label): (4255300,)
--------------------------------------------------
🎯 Hoàn thành Cell 1! Dữ liệu đã sẵn sàng trên RAM.


In [7]:
from scipy import stats
import pandas as pd
import numpy as np

print("⏳ Bắt đầu cắt cửa sổ trượt và trích xuất đặc trưng... (Quá trình này có thể mất 1-2 phút)")

# 1. Định nghĩa tần số lấy mẫu (Hz)
fs_eda, fs_temp = 4, 4
fs_bvp = 64
fs_acc = 32
fs_label = 700

# 2. Định nghĩa kích thước cửa sổ theo yêu cầu Task A1
window_physio = 60  # 60 giây cho EDA, BVP, TEMP
window_acc = 5      # 5 giây cho ACC
shift = 0.25        # Trượt 0.25 giây

# 3. Chuyển đổi giây sang số lượng điểm dữ liệu (samples)
shift_eda = int(shift * fs_eda)          # 1 sample
window_eda_samples = window_physio * fs_eda # 240 samples

window_bvp_samples = window_physio * fs_bvp # 3840 samples
shift_bvp = int(shift * fs_bvp)          # 16 samples

window_acc_samples = window_acc * fs_acc    # 160 samples
shift_acc = int(shift * fs_acc)          # 8 samples

window_label_samples = window_physio * fs_label
shift_label = int(shift * fs_label)

# Danh sách lưu kết quả
extracted_features = []

# 4. Vòng lặp cắt cửa sổ (Dùng EDA làm chuẩn vì nó đồng bộ với thời gian nhất)
# Ta chạy từ 0 đến hết mảng EDA, mỗi bước nhảy là shift_eda (1 sample = 0.25s)
total_steps = len(eda) - window_eda_samples

for i in range(0, total_steps, shift_eda):
    
    # --- Tính index cho từng loại tín hiệu ---
    # Vì thời gian (t) của mỗi vòng lặp là: t = i / fs_eda (giây)
    t_start = i / fs_eda
    
    # Cắt mảng EDA (60s)
    start_eda = i
    end_eda = i + window_eda_samples
    eda_window = eda[start_eda:end_eda]
    
    # Cắt mảng BVP (60s)
    start_bvp = int(t_start * fs_bvp)
    end_bvp = start_bvp + window_bvp_samples
    bvp_window = bvp[start_bvp:end_bvp]
    
    # Cắt mảng ACC (Lấy 5 giây CUỐI CÙNG của cửa sổ 60s hiện tại)
    # Vì bài báo tính vận động ở thời điểm hiện tại tương ứng với sinh lý tích lũy
    end_time_acc = t_start + window_physio
    start_time_acc = end_time_acc - window_acc
    
    start_acc = int(start_time_acc * fs_acc)
    end_acc = int(end_time_acc * fs_acc)
    acc_window = acc[start_acc:end_acc]
    
    # Cắt mảng Label (60s) và lấy nhãn xuất hiện nhiều nhất (Mode) trong 60s đó
    start_label = int(t_start * fs_label)
    end_label = start_label + window_label_samples
    label_window = labels[start_label:end_label]
    
    # Bỏ qua nếu dữ liệu bị lỗi kích thước ở cuối mảng
    if len(label_window) == 0: continue
    majority_label = stats.mode(label_window, keepdims=True)[0][0]
    
    # --------------------------------------------------
    # 5. TRÍCH XUẤT ĐẶC TRƯNG THỦ CÔNG (HANDCRAFTED FEATURES)
    # --------------------------------------------------
    feature_dict = {
        'EDA_mean': np.mean(eda_window),
        'EDA_std': np.std(eda_window),
        'EDA_min': np.min(eda_window),
        'EDA_max': np.max(eda_window),
        
        'BVP_mean': np.mean(bvp_window),
        'BVP_std': np.std(bvp_window),
        
        # Gia tốc kế tính theo công thức Vector độ lớn (Magnitude)
        'ACC_mean': np.mean(np.sqrt(acc_window[:,0]**2 + acc_window[:,1]**2 + acc_window[:,2]**2)),
        
        'Label_Original': majority_label
    }
    
    extracted_features.append(feature_dict)
    
    # In tiến độ cho mỗi 10,000 cửa sổ
    if len(extracted_features) % 10000 == 0:
        print(f" Đã xử lý {len(extracted_features)} cửa sổ...")

# 6. Chuyển list thành bảng Pandas DataFrame
df_features = pd.DataFrame(extracted_features)

print("✅ Hoàn thành Cell 2!")
print("Kích thước bảng dữ liệu đặc trưng:", df_features.shape)
display(df_features.head())

⏳ Bắt đầu cắt cửa sổ trượt và trích xuất đặc trưng... (Quá trình này có thể mất 1-2 phút)
 Đã xử lý 10000 cửa sổ...
 Đã xử lý 20000 cửa sổ...
✅ Hoàn thành Cell 2!
Kích thước bảng dữ liệu đặc trưng: (24076, 8)


,EDA_mean,EDA_std,EDA_min,EDA_max,BVP_mean,BVP_std,ACC_mean,Label_Original
0,1.232191,0.101280,0.934426,1.616094,0.246159,78.560770,63.354278,0
1,1.231727,0.101963,0.934426,1.616094,0.221703,78.551522,63.508017,0
2,1.231310,0.102598,0.934426,1.616094,0.097911,78.503150,63.441418,0
3,1.231358,0.102498,0.934426,1.616094,-0.190531,78.432154,63.374014,0
4,1.231278,0.102660,0.934426,1.616094,-0.368448,78.412766,63.279628,0


In [8]:
import pandas as pd
import numpy as np

print("⏳ Đang chuẩn bị dữ liệu cho 2 bài toán phân lớp...")

# 1. LOẠI BỎ DỮ LIỆU NHIỄU (CLEANING)
# Trong WESAD, nhãn 0 là lúc cởi/mặc thiết bị (không dùng được)
# Nhãn 1: Baseline, 2: Stress, 3: Amusement (Vui vẻ), 4: Meditation (Thiền)
df_clean = df_features[df_features['Label_Original'].isin([1, 2, 3, 4])].copy()

# ==========================================
# BÀI TOÁN 1: Phân loại 3 lớp (3-class)
# Chỉ lấy các dòng có nhãn 1, 2, 3
# ==========================================
df_3_class = df_clean[df_clean['Label_Original'].isin([1, 2, 3])].copy()

# Tách Đặc trưng (X) và Nhãn (y)
X_3class = df_3_class.drop(columns=['Label_Original'])
y_3class = df_3_class['Label_Original']


# ==========================================
# BÀI TOÁN 2: Phân loại nhị phân (2-class)
# Stress = 1 (Từ nhãn gốc 2)
# Non-stress = 0 (Gộp từ nhãn gốc 1, 3, 4)
# ==========================================
df_2_class = df_clean.copy()

# Hàm biến đổi (Map): Nếu nhãn gốc == 2 thì trả về 1, còn lại trả về 0
df_2_class['Label_Binary'] = df_2_class['Label_Original'].apply(lambda x: 1 if x == 2 else 0)

# Tách Đặc trưng (X) và Nhãn (y)
X_2class = df_2_class.drop(columns=['Label_Original', 'Label_Binary'])
y_2class = df_2_class['Label_Binary']

# ==========================================
# IN THỐNG KÊ ĐỂ KIỂM TRA MỨC ĐỘ MẤT CÂN BẰNG DỮ LIỆU
# ==========================================
print("-" * 40)
print("📊 THỐNG KÊ NHÃN - TASK 3-CLASS")
print(y_3class.value_counts().rename(index={1: '1 (Baseline)', 2: '2 (Stress)', 3: '3 (Amusement)'}))

print("-" * 40)
print("📊 THỐNG KÊ NHÃN - TASK 2-CLASS")
print(y_2class.value_counts().rename(index={0: '0 (Non-stress)', 1: '1 (Stress)'}))
print("-" * 40)

print("✅ Hoàn thành Cell 3! Bộ biến X_3class, y_3class và X_2class, y_2class đã sẵn sàng.")

⏳ Đang chuẩn bị dữ liệu cho 2 bài toán phân lớp...
----------------------------------------
📊 THỐNG KÊ NHÃN - TASK 3-CLASS
Label_Original
1 (Baseline)     4576
2 (Stress)       2460
3 (Amusement)    1448
Name: count, dtype: int64
----------------------------------------
📊 THỐNG KÊ NHÃN - TASK 2-CLASS
Label_Binary
0 (Non-stress)    9096
1 (Stress)        2460
Name: count, dtype: int64
----------------------------------------
✅ Hoàn thành Cell 3! Bộ biến X_3class, y_3class và X_2class, y_2class đã sẵn sàng.


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore') # Tắt các cảnh báo lặt vặt cho màn hình gọn gàng

print("🚀 BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH (MODELING)\n")

# 1. Khởi tạo 3 mô hình giống hệt trong bài báo khoa học
models = {
    "Random Forest (Rừng ngẫu nhiên)": RandomForestClassifier(n_estimators=100, random_state=42),
    "AdaBoost (Tăng cường)": AdaBoostClassifier(n_estimators=50, random_state=42),
    "LDA (Phân tích phân biệt tuyến tính)": LinearDiscriminantAnalysis()
}

# 2. Xây dựng hàm dùng chung để đánh giá (tránh viết code lặp lại 2 lần)
def train_and_evaluate(X, y, task_name):
    print(f"==================================================")
    print(f"🎯 KẾT QUẢ CHO BÀI TOÁN: {task_name}")
    print(f"==================================================")
    
    # Chia tập dữ liệu thành 80% để Học (Train) và 20% để Thi (Test)
    # stratify=y giúp giữ nguyên tỷ lệ mất cân bằng của nhãn trong tập test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    for name, model in models.items():
        # Bước 1: Máy tính "Học" từ tập Train
        model.fit(X_train, y_train)
        
        # Bước 2: Máy tính "Làm bài thi" trên tập Test
        y_pred = model.predict(X_test)
        
        # Bước 3: Chấm điểm (Sử dụng Accuracy, Precision, Recall, F1-score từ Chương 6)
        print(f"\n▶️ Mô hình: {name}")
        print(f"Độ chính xác tổng thể (Accuracy): {accuracy_score(y_test, y_pred)*100:.2f}%")
        
        # classification_report in ra bảng chi tiết F1-score cho từng nhãn
        print("Bảng báo cáo chi tiết:")
        print(classification_report(y_test, y_pred))

# 3. CHẠY THỬ NGHIỆM CHO CẢ 2 BÀI TOÁN
# Đưa dữ liệu 2-class (đã tách ở Cell 3) vào đánh giá
train_and_evaluate(X_2class, y_2class, "2-CLASS (Stress = 1 vs Non-Stress = 0)")

print("\n" + "*"*50 + "\n")

# Đưa dữ liệu 3-class (đã tách ở

🚀 BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH (MODELING)

🎯 KẾT QUẢ CHO BÀI TOÁN: 2-CLASS (Stress = 1 vs Non-Stress = 0)

▶️ Mô hình: Random Forest (Rừng ngẫu nhiên)
Độ chính xác tổng thể (Accuracy): 100.00%
Bảng báo cáo chi tiết:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1820
           1       1.00      1.00      1.00       492

    accuracy                           1.00      2312
   macro avg       1.00      1.00      1.00      2312
weighted avg       1.00      1.00      1.00      2312


▶️ Mô hình: AdaBoost (Tăng cường)
Độ chính xác tổng thể (Accuracy): 100.00%
Bảng báo cáo chi tiết:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1820
           1       1.00      1.00      1.00       492

    accuracy                           1.00      2312
   macro avg       1.00      1.00      1.00      2312
weighted avg       1.00      1.00      1.00      2312


▶️ Mô hình: LDA (Phân tích phân

# --- PHẦN 2: CHẠY PIPELINE TỔNG HỢP & LOSO-CV CHO TOÀN BỘ WESAD ---


In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import f1_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

print("🚀 BẮT ĐẦU CHẠY PIPELINE TỔNG HỢP (TASK A1)...\n")

# =====================================================================
# PHẦN 1: HÀM TRÍCH XUẤT ĐẶC TRƯNG CHO 1 NGƯỜI (Gộp Cell 1 & Cell 2)
# =====================================================================
def process_single_subject(file_path, subject_id):
    with open(file_path, 'rb') as file:
        data = pickle.load(file, encoding='latin1')
    
    wrist = data['signal']['wrist']
    acc, bvp, eda = wrist['ACC'], wrist['BVP'], wrist['EDA']
    labels = data['label']
    
    # Cấu hình cửa sổ
    fs_eda, fs_bvp, fs_acc, fs_label = 4, 64, 32, 700
    window_physio, window_acc, shift = 60, 5, 0.25
    
    shift_eda = int(shift * fs_eda)
    window_eda_samples = window_physio * fs_eda
    window_bvp_samples = window_physio * fs_bvp
    window_acc_samples = window_acc * fs_acc
    window_label_samples = window_physio * fs_label
    
    features = []
    total_steps = len(eda) - window_eda_samples
    
    for i in range(0, total_steps, shift_eda):
        t_start = i / fs_eda
        
        # Cắt mảng
        eda_w = eda[i : i + window_eda_samples]
        bvp_w = bvp[int(t_start * fs_bvp) : int(t_start * fs_bvp) + window_bvp_samples]
        
        # ACC lấy 5s cuối của cửa sổ 60s
        end_time_acc = t_start + window_physio
        start_time_acc = end_time_acc - window_acc
        acc_w = acc[int(start_time_acc * fs_acc) : int(end_time_acc * fs_acc)]
        
        # Label
        label_w = labels[int(t_start * fs_label) : int(t_start * fs_label) + window_label_samples]
        if len(label_w) == 0: continue
        majority_label = stats.mode(label_w, keepdims=True)[0][0]
        
        # Chỉ lấy nhãn 1, 2, 3, 4 (Bỏ qua nhiễu 0)
        if majority_label not in [1, 2, 3, 4]:
            continue
            
        features.append({
            'Subject_ID': subject_id, # Đánh dấu dòng này là của ai
            'EDA_mean': np.mean(eda_w),
            'EDA_std': np.std(eda_w),
            'BVP_mean': np.mean(bvp_w),
            'BVP_std': np.std(bvp_w),
            'ACC_mean': np.mean(np.sqrt(acc_w[:,0]**2 + acc_w[:,1]**2 + acc_w[:,2]**2)),
            'Label_Original': majority_label
        })
        
    return pd.DataFrame(features)

# =====================================================================
# PHẦN 2: DUYỆT QUA TẤT CẢ CÁC THƯ MỤC S2, S3... ĐỂ GOM DỮ LIỆU
# =====================================================================
base_folder = '../data/raw/WESAD'
all_subjects_df = []

print("⏳ Đang trích xuất đặc trưng cho từng Subject (Quá trình này mất khoảng 10-15 phút)...")
for root, dirs, files in os.walk(base_folder):
    for file in files:
        if file.endswith('.pkl'):
            subject_id = file.split('.')[0] # Lấy chữ 'S2', 'S3'...
            full_path = os.path.join(root, file)
            print(f"  -> Đang xử lý: {subject_id} ...")
            
            df_subject = process_single_subject(full_path, subject_id)
            all_subjects_df.append(df_subject)

# Ghép tất cả lại thành 1 bảng DataFrame siêu to
df_master = pd.concat(all_subjects_df, ignore_index=True)
print(f"✅ Đã gom xong! Tổng số điểm dữ liệu: {df_master.shape[0]} dòng.")

# =====================================================================
# PHẦN 3: ĐÁNH GIÁ MÔ HÌNH BẰNG LOSO-CV (Giống Paper)
# =====================================================================
# Chuẩn bị 2 bài toán
# Task 2-class (1 = Stress, 0 = Non-stress)
df_2_class = df_master.copy()
df_2_class['Label_Binary'] = df_2_class['Label_Original'].apply(lambda x: 1 if x == 2 else 0)

# Khởi tạo mô hình Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

subjects = df_2_class['Subject_ID'].unique()
loso_f1_scores = []
loso_accuracies = []

print("\n🚀 BẮT ĐẦU CHẠY LOSO-CV CHO BÀI TOÁN 2-CLASS (Random Forest)...")
for test_sub in subjects:
    # Tách tập Train (Tất cả mọi người trừ test_sub) và Test (Chỉ mình test_sub)
    train_data = df_2_class[df_2_class['Subject_ID'] != test_sub]
    test_data = df_2_class[df_2_class['Subject_ID'] == test_sub]
    
    X_train = train_data.drop(columns=['Subject_ID', 'Label_Original', 'Label_Binary'])
    y_train = train_data['Label_Binary']
    
    X_test = test_data.drop(columns=['Subject_ID', 'Label_Original', 'Label_Binary'])
    y_test = test_data['Label_Binary']
    
    # Train và Test
    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)
    
    # Tính điểm
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0) # F1 cho lớp Stress
    
    loso_accuracies.append(acc)
    loso_f1_scores.append(f1)
    
    print(f"  - Test trên {test_sub} | Điểm F1 (Stress): {f1*100:.2f}% | Accuracy: {acc*100:.2f}%")

print("="*50)
print("🏆 KẾT QUẢ CUỐI CÙNG TASK A1 (WESAD BASELINE - LOSO CV):")
print(f"⭐ Trung bình F1-Score (Stress) : {np.mean(loso_f1_scores)*100:.2f}%")
print(f"⭐ Trung bình Accuracy tổng thể : {np.mean(loso_accuracies)*100:.2f}%")
print("="*50)


🚀 BẮT ĐẦU CHẠY PIPELINE TỔNG HỢP (TASK A1)...

⏳ Đang trích xuất đặc trưng cho từng Subject (Quá trình này mất khoảng 10-15 phút)...
  -> Đang xử lý: S10 ...
  -> Đang xử lý: S11 ...
  -> Đang xử lý: S13 ...
  -> Đang xử lý: S14 ...
  -> Đang xử lý: S15 ...
  -> Đang xử lý: S16 ...
  -> Đang xử lý: S17 ...
  -> Đang xử lý: S2 ...
  -> Đang xử lý: S3 ...
  -> Đang xử lý: S4 ...
  -> Đang xử lý: S5 ...
  -> Đang xử lý: S6 ...
  -> Đang xử lý: S7 ...
  -> Đang xử lý: S8 ...
  -> Đang xử lý: S9 ...
✅ Đã gom xong! Tổng số điểm dữ liệu: 179817 dòng.

🚀 BẮT ĐẦU CHẠY LOSO-CV CHO BÀI TOÁN 2-CLASS (Random Forest)...
  - Test trên S10 | Điểm F1 (Stress): 5.86% | Accuracy: 76.49%
  - Test trên S11 | Điểm F1 (Stress): 49.81% | Accuracy: 69.70%
  - Test trên S13 | Điểm F1 (Stress): 53.14% | Accuracy: 61.23%
  - Test trên S14 | Điểm F1 (Stress): 0.00% | Accuracy: 77.63%
  - Test trên S15 | Điểm F1 (Stress): 22.64% | Accuracy: 80.14%
  - Test trên S16 | Điểm F1 (Stress): 0.22% | Accuracy: 77.69%
  - T